# 🏺 Hieroglyph NLP Pipeline
### Transliteration → Dictionary → spaCy → Arabic Translation → Sentiment
---
> Running on **Google Colab**

## ▶ Cell 0 — Install Dependencies

In [1]:
import subprocess, sys

def pip(pkg):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

for pkg in ['transformers', 'sentencepiece', 'torch', 'requests', 'spacy', 'scipy']:
    pip(pkg)

# spaCy English model
subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'], check=True)

print('✅ All dependencies installed')


✅ All dependencies installed


## 📁 Cell 1 — Upload Dataset Files

In [3]:
from google.colab import files

print('Upload your CSV files:')
print('  1. gardiner CSV (Update_gardiner_sign.csv)')
print('  2. egyptian_dictionary.csv')
print('  3. intention_dataset.csv')
print()

uploaded = files.upload()

print()
print('Uploaded:')
for fname in uploaded:
    with open(f'/content/{fname}', 'wb') as f:
        f.write(uploaded[fname])
    print(f'  ✅ {fname} saved to /content/')


Upload your CSV files:
  1. gardiner CSV (Update_gardiner_sign.csv)
  2. egyptian_dictionary.csv
  3. intention_dataset.csv



Saving egyptian_dictionary.csv to egyptian_dictionary.csv
Saving intention_dataset.csv to intention_dataset.csv
Saving Update_gardiner_sign.csv to Update_gardiner_sign.csv

Uploaded:
  ✅ egyptian_dictionary.csv saved to /content/
  ✅ intention_dataset.csv saved to /content/
  ✅ Update_gardiner_sign.csv saved to /content/


In [4]:
# ── Test Cell 1 ────────────────────────────────────────────────
import os
print('Files in /content/:')
for f in os.listdir('/content'):
    if f.endswith('.csv'):
        size = os.path.getsize(f'/content/{f}')
        print(f'  {f} ({size:,} bytes)')


Files in /content/:
  Update_gardiner_sign.csv (22,069 bytes)
  egyptian_dictionary.csv (45,188 bytes)
  intention_dataset.csv (23,552 bytes)


## 📖 Cell 2 — Gardiner Sign List

In [5]:
import pandas as pd

DATASET_PATH = '/content/Update_gardiner_sign.csv'
df = pd.read_csv(DATASET_PATH)

GARDINER_MAP = {}
for _, row in df.iterrows():
    code = str(row['code']).strip().lower()
    if not code or code == 'nan':
        continue
    GARDINER_MAP[code] = {
        'phonetic': str(row['phonetic']).strip() if pd.notna(row['phonetic']) else '',
        'meaning' : str(row['meaning']).strip()  if pd.notna(row['meaning'])  else '',
        'unicode' : str(row['unicode']).strip()   if pd.notna(row['unicode'])  else '',
    }

print(f'✅ Gardiner Sign List loaded: {len(GARDINER_MAP)} signs')
print(f'   Signs with phonetic: {sum(1 for v in GARDINER_MAP.values() if v["phonetic"])}')


✅ Gardiner Sign List loaded: 819 signs
   Signs with phonetic: 224


In [6]:
# ── Test Cell 2 ────────────────────────────────────────────────
print('Sample signs:')
for code in ['g17', 'n35', 'd21', 'o1', 'n5', 'f35', 'r4']:
    if code in GARDINER_MAP:
        v = GARDINER_MAP[code]
        print(f'  {code.upper():5} -> phonetic: {v["phonetic"]:8} | {v["meaning"]:30} {v["unicode"]}')
    else:
        print(f'  {code.upper():5} -> NOT FOUND')


Sample signs:
  G17   -> phonetic: m        | Owl                            𓅓
  N35   -> phonetic: n        | Water ripple                   𓈖
  D21   -> phonetic: r        | Mouth                          𓂋
  O1    -> phonetic: pr       | House plan                     𓉐
  N5    -> phonetic: ra       | Sun disk                       𓇳
  F35   -> phonetic: nfr      | Heart and windpipe             𓄤
  R4    -> phonetic: Htp      | Bread loaf on mat              𓊵


## 📚 Cell 3 — Egyptian Dictionary

In [7]:
import pandas as pd
from collections import defaultdict

DICT_PATH = '/content/egyptian_dictionary.csv'
df_dict = pd.read_csv(DICT_PATH)

_raw = defaultdict(list)
for _, row in df_dict.iterrows():
    key = str(row['transliteration']).strip()
    val = str(row['english']).strip()
    if not key or key == 'nan' or not val or val == 'nan':
        continue
    if val not in _raw[key]:
        _raw[key].append(val)

EGYPTIAN_DICT = dict(_raw)

print(f'✅ Egyptian Dictionary loaded: {len(EGYPTIAN_DICT)} entries')


✅ Egyptian Dictionary loaded: 1352 entries


In [8]:
# ── Test Cell 3 ────────────────────────────────────────────────
test_words = ['nfr', 'ra', 'Htp', 'rn', 'nTr', 'pr', 'zA']
print('Dictionary sample:')
for w in test_words:
    meanings = EGYPTIAN_DICT.get(w, ['NOT FOUND'])
    print(f'  {w:10} -> {" | ".join(meanings[:3])}')


Dictionary sample:
  nfr        -> be good / beautiful / perfect | good / beautiful / perfect | perfect / good
  ra         -> sun god Ra / Re
  Htp        -> offering / peace / satisfaction | be satisfied / rest | offer / be at peace
  rn         -> name / soul name
  nTr        -> god
  pr         -> house / estate
  zA         -> NOT FOUND


## ⚙️ Cell 4 — Transliteration Engine + spaCy Sentence Builder

In [9]:
import re
import spacy

print('Loading spaCy...')
nlp_spacy = spacy.load('en_core_web_sm')
print('✅ spaCy loaded')


def extract_core_meaning(meanings) -> str:
    if isinstance(meanings, list):
        text = meanings[0]
    else:
        text = str(meanings)
    first = re.split(r'[/|]', text)[0].strip()
    first = re.sub(r'^be ', '', first).strip()
    return first.lower()


def build_sentence_spacy(core_meanings: list) -> str:
    """
    spaCy POS tagger -> Egyptian grammar rules -> English sentence
    ['name', 'sun'] -> 'my name is sun'
    """
    if not core_meanings:
        return ''
    if len(core_meanings) == 1:
        return core_meanings[0]

    text  = ' '.join(core_meanings)
    doc   = nlp_spacy(text)
    nouns = [t.text for t in doc if t.pos_ in ('NOUN', 'PROPN')]
    verbs = [t.text for t in doc if t.pos_ == 'VERB']
    adjs  = [t.text for t in doc if t.pos_ == 'ADJ']

    possession = {'name','son','daughter','house','heart',
                  'brother','sister','father','mother','lord'}

    if verbs and nouns:
        return f'{verbs[0]} the {nouns[0]}'
    if len(nouns) >= 2:
        n1, n2 = nouns[0], nouns[1]
        if n1.lower() in possession:
            return f'my {n1} is {n2}'
        return f'{n1} of {n2}'
    if nouns and adjs:
        return f'the {adjs[0]} {nouns[0]}'
    if adjs:
        return f'it is {adjs[0]}'
    return text


def transliterate(gardiner_codes: list) -> dict:
    codes         = [c.lower().strip() for c in gardiner_codes]
    phonetics     = []
    glyphs        = []
    sign_meanings = []
    unknown       = []

    for code in codes:
        if code in GARDINER_MAP:
            info = GARDINER_MAP[code]
            phonetics.append(info['phonetic'])
            glyphs.append(info['unicode'])
            sign_meanings.append(info['meaning'])
        else:
            phonetics.append('?')
            glyphs.append('□')
            sign_meanings.append('unknown')
            unknown.append(code)

    ph_clean = [p for p in phonetics if p and p != '?']
    full     = ''.join(ph_clean)

    # Greedy multi-phoneme matching
    token_results = []
    core_meanings = []
    i = 0
    while i < len(ph_clean):
        matched = False
        for length in range(min(4, len(ph_clean) - i), 0, -1):
            combined = ''.join(ph_clean[i:i+length])
            if combined in EGYPTIAN_DICT:
                meanings_list = EGYPTIAN_DICT[combined]
                meaning_str   = ' | '.join(meanings_list)
                core          = extract_core_meaning(meanings_list)
                token_results.append({'phonetic':combined,'meaning':meaning_str,'core':core,'found':True})
                core_meanings.append(core)
                i += length
                matched = True
                break
        if not matched:
            ph = ph_clean[i]
            token_results.append({'phonetic':ph,'meaning':f'[{ph}]','core':ph,'found':False})
            i += 1

    # Sliding window assembly
    assembly_candidates = []
    if full: assembly_candidates.append(full)
    for size in range(len(ph_clean), 0, -1):
        for start in range(len(ph_clean) - size + 1):
            window = ''.join(ph_clean[start:start+size])
            if window and window not in assembly_candidates:
                assembly_candidates.append(window)

    found_words = []
    for candidate in assembly_candidates:
        if candidate in EGYPTIAN_DICT:
            meanings_list = EGYPTIAN_DICT[candidate]
            meaning_str   = ' | '.join(meanings_list)
            found_words.append({
                'transliteration': candidate,
                'meaning'        : meaning_str,
                'confidence'     : 'high' if candidate == full else 'partial',
            })

    high = [w for w in found_words if w['confidence'] == 'high']
    part = [w for w in found_words if w['confidence'] == 'partial']
    best = high[0] if high else (part[0] if part else None)
    sentence = build_sentence_spacy(core_meanings)

    return {
        'input_codes'  : codes,
        'glyphs'       : glyphs,
        'glyph_str'    : ' '.join(glyphs),
        'per_sign'     : list(zip(codes, phonetics, sign_meanings)),
        'phonetics'    : phonetics,
        'phonetic_str' : ' '.join(ph_clean),
        'assembled'    : full,
        'token_results': token_results,
        'found_words'  : found_words,
        'unknown_codes': unknown,
        'best_meaning' : best['meaning'] if best else None,
        'sentence'     : sentence,
        'core_meanings': core_meanings,
    }

print('✅ Transliteration engine ready')


Loading spaCy...
✅ spaCy loaded
✅ Transliteration engine ready


In [10]:
# ── Test Cell 4 ────────────────────────────────────────────────
test_cases = [
    (['o1'],              'pr   -> house'),
    (['f35'],             'nfr  -> beautiful'),
    (['n5', 'r8'],        'ra nTr -> sun god'),
    (['d21', 'n35', 'n5'],'rn ra  -> my name is sun'),
    (['o4', 'n5'],        'zA ra  -> son of Ra'),
    (['r4', 'x8', 'a42'], 'Htp di nswt -> offering'),
]

for codes, label in test_cases:
    r = transliterate(codes)
    print(f'  {label}')
    print(f'    Tokens  : {[(t["phonetic"], t["core"]) for t in r["token_results"]]}')
    print(f'    Sentence: {r["sentence"]}')
    print()


  pr   -> house
    Tokens  : [('pr', 'house')]
    Sentence: house

  nfr  -> beautiful
    Tokens  : [('nfr', 'good')]
    Sentence: good

  ra nTr -> sun god
    Tokens  : [('ra', 'sun god ra'), ('nTr', 'god')]
    Sentence: sun of god

  rn ra  -> my name is sun
    Tokens  : [('rn', 'name'), ('ra', 'sun god ra')]
    Sentence: my name is sun

  zA ra  -> son of Ra
    Tokens  : [('H', 'H'), ('ra', 'sun god ra')]
    Sentence: sun god ra

  Htp di nswt -> offering
    Tokens  : [('Htp', 'offering'), ('di', 'give')]
    Sentence: offering give



## 🌍 Cell 5 — Arabic Translation
> Model: `Helsinki-NLP/opus-mt-en-ar`
> متخصص إنجليزي → عربي، دقيق وخفيف

In [11]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

print('Loading NLLB translation model...')
print('facebook/nllb-200-distilled-600M')

NLLB_MODEL  = 'facebook/nllb-200-distilled-600M'
ar_tokenizer = AutoTokenizer.from_pretrained(NLLB_MODEL)
ar_model     = AutoModelForSeq2SeqLM.from_pretrained(NLLB_MODEL)
ar_model.eval()

print('✅ NLLB model loaded')


def translate_to_arabic(english_text: str) -> str:
    if not english_text or english_text.startswith('['):
        return ''
    try:
        import torch
        inputs = ar_tokenizer(
            english_text,
            return_tensors='pt',
            truncation=True,
            max_length=128,
        )
        with torch.no_grad():
            translated = ar_model.generate(
                **inputs,
                forced_bos_token_id = ar_tokenizer.convert_tokens_to_ids('arb_Arab'),
                max_new_tokens      = 60,
                num_beams           = 5,
                no_repeat_ngram_size= 3,
                repetition_penalty  = 1.5,
            )
        result = ar_tokenizer.decode(translated[0], skip_special_tokens=True)
        return result
    except Exception as e:
        return f'[error: {e}]'

print('✅ translate_to_arabic() ready')

Loading NLLB translation model...
facebook/nllb-200-distilled-600M


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

✅ NLLB model loaded
✅ translate_to_arabic() ready


In [12]:
# ── Test Cell 5 ────────────────────────────────────────────────
test_translations = [
    'my name is sun',
    'son of Ra',
    'offering which the king gives',
    'the good god',
    'house of god',
    'it is beautiful',
    'give the offering',
]

print('English -> Arabic translations:')
print()
for text in test_translations:
    arabic = translate_to_arabic(text)
    print(f'  EN: {text}')
    print(f'  AR: {arabic}')
    print()


English -> Arabic translations:

  EN: my name is sun
  AR: اسمي هو الشمس

  EN: son of Ra
  AR: ابن را

  EN: offering which the king gives
  AR: والتي يقدمها الملك تضحية

  EN: the good god
  AR: الإله الجيد

  EN: house of god
  AR: بيت الإله

  EN: it is beautiful
  AR: إنها جميلة

  EN: give the offering
  AR: إعطاء العرض



## 💭 Cell 6 — Sentiment Analysis
> Model: `cardiffnlp/twitter-roberta-base-sentiment`
> اتدرب على **124 مليون تويت** — positive / negative / neutral

In [13]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
import numpy as np
import pandas as pd

print('Loading Twitter RoBERTa sentiment model...')
SENTIMENT_MODEL_NAME = 'cardiffnlp/twitter-roberta-base-sentiment'
sent_tokenizer = AutoTokenizer.from_pretrained(SENTIMENT_MODEL_NAME)
sent_model     = AutoModelForSequenceClassification.from_pretrained(SENTIMENT_MODEL_NAME)
sent_model.eval()
SENTIMENT_LABELS = ['negative', 'neutral', 'positive']
print('✅ Sentiment model loaded')


def analyze_sentiment(text: str) -> tuple:
    if not text or text.startswith('['):
        return 'neutral', 0.5
    try:
        inputs = sent_tokenizer(text, return_tensors='pt', truncation=True, max_length=128)
        with torch.no_grad():
            outputs = sent_model(**inputs)
        scores     = torch.softmax(outputs.logits, dim=1).numpy()[0]
        best_idx   = int(np.argmax(scores))
        return SENTIMENT_LABELS[best_idx], round(float(scores[best_idx]), 3)
    except:
        return 'neutral', 0.5


# ── Load Intention Dataset from CSV ──────────────────────────────────
INTENTION_PATH = '/content/intention_dataset.csv'
df_intent = pd.read_csv(INTENTION_PATH)

# بناء الـ dict: intention -> {ar, keywords_set}
INTENTION_MAP = {}
for _, row in df_intent.iterrows():
    intent_en = str(row['intention_en']).strip()
    intent_ar = str(row['intention_ar']).strip()
    keywords  = [kw.strip().lower() for kw in str(row['keywords']).split(',')]
    INTENTION_MAP[intent_en] = {
        'arabic'  : intent_ar,
        'keywords': set(keywords),
    }

print(f'✅ Intention dataset loaded: {len(INTENTION_MAP)} intentions')


def detect_intention(text: str, phonetics: str = '') -> tuple:
    """
    بيبحث في الـ intention dataset
    Returns: (intention_en, intention_ar, scores_dict)
    """
    combined = (text + ' ' + phonetics).lower()
    words    = set(combined.split())
    scores   = {}
    for intent, data in INTENTION_MAP.items():
        hits = len(words & data['keywords'])
        if hits > 0:
            scores[intent] = hits
    if not scores:
        return 'descriptive', 'وصفي', {}
    best_intent = max(scores, key=scores.get)
    best_ar     = INTENTION_MAP[best_intent]['arabic']
    return best_intent, best_ar, scores

print('✅ detect_intention() ready')


Loading Twitter RoBERTa sentiment model...


config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Sentiment model loaded
✅ Intention dataset loaded: 139 intentions
✅ detect_intention() ready


In [15]:
# ── Test Cell 6 ────────────────────────────────────────────────
test_sentiments = [
    'my name is sun',
    'offering of the king',
    'god of the beautiful house',
    'death and darkness',
    'beloved son of Ra',
    'give life and peace',
]

print('Sentiment Analysis (Twitter RoBERTa):')
print()
for text in test_sentiments:
    label, score = analyze_sentiment(text)
    intent_en, intent_ar, detail = detect_intention(text)  # ← 3 قيم
    emoji = '😊' if label == 'positive' else '😞' if label == 'negative' else '😐'
    print(f'  {emoji} [{label:8}] ({score}) | {text}')
    print(f'     Intention: {intent_en} — {intent_ar}')
    print()

Sentiment Analysis (Twitter RoBERTa):

  😐 [neutral ] (0.609) | my name is sun
     Intention: resurrection — البعث والقيامة

  😐 [neutral ] (0.791) | offering of the king
     Intention: funerary_formula — صيغة جنائزية

  😊 [positive] (0.689) | god of the beautiful house
     Intention: hymn — تسبيح وترنيم

  😞 [negative] (0.518) | death and darkness
     Intention: sorrow — الحزن والألم

  😊 [positive] (0.728) | beloved son of Ra
     Intention: royal_titulary — ألقاب ملكية

  😊 [positive] (0.764) | give life and peace
     Intention: thanksgiving — شكر وامتنان



## 🚀 Cell 7 — Full NLP Pipeline

In [16]:
def run_nlp_pipeline(gardiner_codes: list, verbose: bool = True) -> dict:
    trans        = transliterate(gardiner_codes)
    glyph_str    = trans['glyph_str']
    phonetic_str = trans['phonetic_str']
    found_words  = trans['found_words']
    best_meaning = trans['best_meaning']
    sentence     = trans['sentence']

    if best_meaning:
        english = best_meaning
        method  = 'dictionary'
    elif sentence:
        english = sentence
        method  = 'spacy-nlp'
    else:
        english = f'[unknown: {trans["assembled"]}]'
        method  = 'none'

    arabic           = translate_to_arabic(sentence if sentence else english)
    sentiment_text   = (sentence + ' ' + (best_meaning or '')).strip()
    sentiment, score = analyze_sentiment(sentiment_text)
    intention_en, intention_ar, intent_detail = detect_intention(sentiment_text, phonetic_str)

    result = {
        'input'        : gardiner_codes,
        'glyphs'       : glyph_str,
        'per_sign'     : trans['per_sign'],
        'phonetics'    : phonetic_str,
        'assembled'    : trans['assembled'],
        'token_results': trans['token_results'],
        'found_words'  : found_words,
        'sentence'     : sentence,
        'english'      : english,
        'arabic'       : arabic,
        'trans_method' : method,
        'sentiment'    : sentiment,
        'sent_score'   : score,
        'intention_en' : intention_en,
        'intention_ar' : intention_ar,
        'intent_detail': intent_detail,
    }

    if verbose:
        print('=' * 62)
        print('  🏺  HIEROGLYPH NLP PIPELINE')
        print('=' * 62)
        print(f'  Input codes  : {gardiner_codes}')
        print(f'  Glyphs       : {glyph_str}')
        print()
        print('  Per sign:')
        for code, ph, meaning in trans['per_sign']:
            ph_str = ph if ph else '(det.)'
            print(f'    {code.upper():6} -> {ph_str:10} | {meaning}')
        print()
        print('  Tokens:')
        for t in trans['token_results']:
            icon = 'OK' if t['found'] else '?'
            print(f'    [{icon}] {t["phonetic"]:8} -> {t["meaning"][:40]}')
        print()
        if found_words:
            print('  Word matches:')
            for w in found_words[:3]:
                print(f'    [{w["confidence"]:7}] {w["transliteration"]:12} = {w["meaning"][:40]}')
        print()
        print(f'  Sentence     : {sentence}')
        print(f'  English      : {english} [{method}]')
        print(f'  Arabic       : {arabic}')
        print()
        emoji = '😊' if sentiment == 'positive' else '😞' if sentiment == 'negative' else '😐'
        print(f'  Sentiment    : {emoji} {sentiment} (score={score})')
        print(f'  Intention EN : {intention_en}')
        print(f'  Intention AR : {intention_ar}')
        print('=' * 62)

    return result


print('TEST: rn ra')
r = run_nlp_pipeline(['d21', 'n35', 'n5'])

print()
print('TEST: Htp di nsw')
r = run_nlp_pipeline(['r4', 'x8', 'a42'])

print()
print('TEST: ra nTr nfr')
r = run_nlp_pipeline(['n5', 'r8', 'f35'])


TEST: rn ra
  🏺  HIEROGLYPH NLP PIPELINE
  Input codes  : ['d21', 'n35', 'n5']
  Glyphs       : 𓂋 𓈖 𓇳

  Per sign:
    D21    -> r          | Mouth
    N35    -> n          | Water ripple
    N5     -> ra         | Sun disk

  Tokens:
    [OK] rn       -> name / soul name
    [OK] ra       -> sun god Ra / Re

  Word matches:
    [partial] rn           = name / soul name
    [partial] r            = mouth | to / toward / concerning
    [partial] n            = to / for / of

  Sentence     : my name is sun
  English      : name / soul name [dictionary]
  Arabic       : اسمي هو الشمس

  Sentiment    : 😐 neutral (score=0.801)
  Intention EN : rebirth
  Intention AR : التجدد والبعث

TEST: Htp di nsw
  🏺  HIEROGLYPH NLP PIPELINE
  Input codes  : ['r4', 'x8', 'a42']
  Glyphs       : 𓊵 𓏕 𓀯

  Per sign:
    R4     -> Htp        | Bread loaf on mat
    X8     -> di         | Conical bread
    A42    -> (det.)     | Seated king holding flail

  Tokens:
    [OK] Htp      -> offering / peace / sat

## 🧪 Cell 8 — Mock Tests

In [ ]:
MOCK_EXAMPLES = {
    'funerary_text'   : {'description': 'نص جنائزي', 'codes': ['r4','x8','a42','n5','r8','f35','n35','n5']},
    'royal_cartouche' : {'description': 'كارتوش ملكي', 'codes': ['a42','g17','n35','d21','n5']},
    'offering_formula': {'description': 'صيغة قربان', 'codes': ['r4','x8','a42','f35','n35','g43']},
    'simple_word'     : {'description': 'house', 'codes': ['o1']},
    'name_of_ra'      : {'description': 'my name is sun', 'codes': ['d21','n35','n5']},
}

print('Running mock tests...')
print()
for name, ex in MOCK_EXAMPLES.items():
    print(f'  [{name}] {ex["description"]}')
    r = run_nlp_pipeline(ex['codes'], verbose=False)
    print(f'    Glyphs   : {r["glyphs"]}')
    print(f'    Sentence : {r["sentence"]}')
    print(f'    English  : {r["english"]}')
    print(f'    Arabic   : {r["arabic"]}')
    emoji = '😊' if r['sentiment'] == 'positive' else '😞' if r['sentiment'] == 'negative' else '😐'
    print(f'    Sentiment: {emoji} {r["sentiment"]} ({r["sent_score"]})')
    print(f'    Intention: {r["intention"]}')
    print()


## 🔗 Cell 9 — Integration مع الـ CV Model

In [ ]:
def full_pipeline_from_image(image_path: str,
                              cv_predict_fn=None,
                              mock_codes: list = None,
                              verbose: bool = True) -> dict:
    """
    End-to-end: image -> CV -> Gardiner codes -> NLP -> Arabic + Sentiment
    """
    print(f'Processing: {image_path}')
    print()

    if cv_predict_fn is not None:
        cv_results     = cv_predict_fn(image_path, show=False)
        gardiner_codes = [r[1].lower() for r in cv_results]
        confidences    = [r[2] for r in cv_results]
        print(f'  CV detected {len(gardiner_codes)} glyphs')
        print(f'  Codes: {gardiner_codes}')
        print(f'  Conf : {[round(c,2) for c in confidences]}')
    elif mock_codes is not None:
        gardiner_codes = [c.lower() for c in mock_codes]
        print(f'  Using mock codes: {gardiner_codes}')
    else:
        raise ValueError('Provide cv_predict_fn or mock_codes')

    if not gardiner_codes:
        return {'error': 'No glyphs detected'}

    print()
    nlp_result = run_nlp_pipeline(gardiner_codes, verbose=verbose)

    return {
        'image'    : image_path,
        'n_glyphs' : len(gardiner_codes),
        'codes'    : gardiner_codes,
        'glyphs'   : nlp_result['glyphs'],
        'phonetics': nlp_result['phonetics'],
        'sentence' : nlp_result['sentence'],
        'english'  : nlp_result['english'],
        'arabic'   : nlp_result['arabic'],
        'sentiment': nlp_result['sentiment'],
        'intention': nlp_result['intention'],
    }


# Demo
print('DEMO:')
result = full_pipeline_from_image(
    image_path = 'stela.jpg',
    mock_codes = ['r4', 'x8', 'a42', 'f35', 'n35', 'n5', 'r8'],
    verbose    = True,
)
print()
print('FINAL SUMMARY:')
for k, v in result.items():
    print(f'  {k:12}: {v}')
